<a href="https://colab.research.google.com/github/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/W13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bias Correction for MUSICA Simulation with AutoML

**Lab Duration:** ~1-2 hours

## Overview

In this lab, you will learn how to apply **AutoML** techniques to perform bias correction of **PM2.5** concentrations — improving the accuracy of model simulations by learning systematic discrepancies between model outputs and observations.

We will learn how to use ObsAQ to extract observational PM2.5 data and work with PM2.5 data derived from Multi-Scale Infrastructure for Chemistry and Aerosol (MUSICA) simulations. The provided dataset includes hourly MUSICA v0 simulations from 2018-11-20 to 2018-11-30. By comparing modelled PM2.5 with observational data, AutoML frameworks can automatically select and optimize machine learning models to correct biases across different regions and time periods.

## Objective

The objective of this lab is to develop an **AutoML-based bias correction model** for PM2.5 that:
- Learns the relationship between model outputs and observations
- Reduces systematic biases in simulated PM2.5 concentrations
- Improves the representation of temporal variability and extreme pollution events
- Provides a scalable and automated framework for air quality model post-processing

---

## Obtain the data from AURN
- Download data within specified **longitude and latitude bounds**.;
- Detailed information on how to use ObsAQ is available on https://github.com/envdes/obsaq

In [ ]:
!pip install obsaq
!pip install requests

In [ ]:
import obsaq
lon_min = -9
lon_max = 1.8
lat_min = 49
lat_max = 61
bounds=[lon_min, lon_max, lat_min, lat_max]

In [ ]:
meta = obsaq.meta()
site_table = meta.get_metadata('aurn')

In [ ]:
final_sites = meta.get_site(bounds=bounds)
final_sites.head(5)

Start to download the selected station data
- **pollutant**: See names of pollutants. Define one pollutant, diverse pollutants or all pollutants to download the data for them. Pollutants can be defined as "PM2.5","PM10","O3","NO","NO2","NOXasNO2" or "SO2".
- **start**: the start date of data to be downloaded.
- **end**: the end date of data to be downloaded.
- **year**: the year of data to be downloaded. Defaults to 2010.
- **output_dir**: the directory to save the downloaded data.
- **download_mode**: "Stream" for saving final and intermediate files while "memory" for only the final file.
- **save_per_site**: whether save files for every station individually.
- **save_merged**: whether save the merged file for all selected data.
- **add_site_id**: whether include site id in the downloaded file.
Download data for every station. Decide whether to download the final merged data for all targeted stations by choosing "save_merged=True/False".

In [ ]:
merged_df = meta.download_sites(
    port="aurn",
    pollutant="PM2.5",
    start="2018-11-20",
    end="2018-11-30",
    output_dir="data/test_pm25",
    download_mode="stream",     
    save_per_site=True,
    save_merged=True,
    add_site_id=True
)

- Download data by **site_id**;
- Detailed information on how to use ObsAQ is available on https://github.com/envdes/obsaq

In [ ]:
import obsaq

meta = obsaq.meta()
site_table = meta.get_metadata('aurn')

site_table.head(5)

In [ ]:
final_sites = meta.get_site(site_id='MAN3')
final_sites.drop_duplicates(subset='site_id')

Start to download the selected station data
- **pollutant**: See names of pollutants. Define one pollutant, diverse pollutants or all pollutants to download the data for them. Pollutants can be defined as "PM2.5","PM10","O3","NO","NO2","NOXasNO2" or "SO2".
- **start**: the start date of data to be downloaded.
- **end**: the end date of data to be downloaded.
- **year**: the year of data to be downloaded. Defaults to 2010.
- **output_dir**: the directory to save the downloaded data.
- **download_mode**: "Stream" for saving final and intermediate files while "memory" for only the final file.
- **save_per_site**: whether save files for every station individually.
- **save_merged**: whether save the merged file for all selected data.
- **add_site_id**: whether include site id in the downloaded file.
Download data for every station. Decide whether to download the final merged data for all targeted stations by choosing "save_merged=True/False".

In [ ]:
meta.download_sites(
    port="aurn",
    pollutant="PM2.5",
    start="2017-11-01",
    end="2017-11-30",
    output_dir="data/test_pm25_siteid",
    download_mode="stream",     
    save_per_site=True,
    save_merged=False,
    add_site_id=True
)

## Extract MUSICAv0 simulation data

In [ ]:
!pip install xarray
!pip install netcdf4 h5netcdf

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

### Data Location
- Original MUSICAv0 simulation: './data/W11.nc': https://github.com/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/data/W11.nc

In [ ]:
BASE_DIR = Path.cwd()
NC_PATH = Path("~/PATH/TO/W11.nc")
OUTPUT_DIR = BASE_DIR / "outputs"
START_TIME = "2018-11-01T10:00:00"
END_TIME = "2018-11-01T12:00:00"
LEV_INDEX = 31
UK_BOUNDS = {
    "lon_min": -8.64999567480067,
    "lon_max": 1.763705376250258,
    "lat_min": 49.86466815672045,
    "lat_max": 60.860787337451533,
}

NC_PATH, START_TIME, END_TIME, LEV_INDEX

In [ ]:
def normalize_longitude(lon_values):
    return (lon_values + 180.0) % 360.0 - 180.0


def extract_uk_pm25(nc_path, start_time, end_time, lev_index):
    ds = xr.open_dataset(nc_path)

    lat = ds["lat"].values
    lon_raw = ds["lon"].values
    lon = normalize_longitude(lon_raw)

    uk_mask = (
        (lon >= UK_BOUNDS["lon_min"])
        & (lon <= UK_BOUNDS["lon_max"])
        & (lat >= UK_BOUNDS["lat_min"])
        & (lat <= UK_BOUNDS["lat_max"])
    )

    uk_ncol = np.flatnonzero(uk_mask)
    pm25 = ds["PM25"].sel(time=slice(start_time, end_time))

    if "lev" in pm25.dims:
        pm25 = pm25.isel(lev=lev_index, ncol=uk_ncol)
        lev_out = lev_index + 1
        lev_value = ds["lev"].isel(lev=lev_index).item()
    else:
        pm25 = pm25.isel(ncol=uk_ncol)
        lev_out = pd.NA
        lev_value = pd.NA

    pm25_df = (
        pm25.to_dataframe(name="PM25")
        .reset_index()
        .assign(
            lev_index=lev_out,
            lev_value=lev_value,
            ncol_index=lambda df: uk_ncol[df["ncol"].to_numpy()],
            grid_lat=lambda df: lat[uk_ncol][df["ncol"].to_numpy()],
            grid_lon=lambda df: lon[uk_ncol][df["ncol"].to_numpy()],
            grid_lon_raw=lambda df: lon_raw[uk_ncol][df["ncol"].to_numpy()],
        )
        .drop(columns=["ncol"])
    )

    ds.close()
    return pm25_df

In [ ]:
pm25_df = extract_uk_pm25(NC_PATH, START_TIME, END_TIME, LEV_INDEX)

print(f"Rows extracted: {len(pm25_df)}")
print(f"Vertical level: {pm25_df['lev_index'].iloc[0]}")
print(pm25_df["time"].drop_duplicates().tolist())
pm25_df.head()

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
output_path = OUTPUT_DIR / "musica_uk_pm25_lev32_2018-11-01_10-12.csv"
pm25_df.to_csv(output_path, index=False)
output_path

## Install AutoML Library (FLAML)

In [ ]:
!pip install flaml

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from flaml import AutoML

### Data Location
- MUSICAv0 simulation: './data/MUSICA_bias_correction_20181120_20181130.csv': https://github.com/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/data/MUSICA_bias_correction_20181120_20181130.csv

In [ ]:
# ==============================
# 1. Read the data
# ==============================

df = pd.read_csv(
    "~/PATH/TO/MUSICA_bias_correction_20181120_20181130.csv"
)

print("Number of rows:", len(df))
print("Available columns:")
print(df.columns.to_list())



In [ ]:
# ==============================
# 2. Define target and baseline model output
# ==============================

obs_col = "PM2.5 (Hourly measured)"   # Observed PM2.5
raw_col = "PM25"                     # Raw MUSICA PM2.5 prediction
time_col = "datetime"


# ==============================
# 3. Select input features
# ==============================

# Basic meteorological variables
meteo_cols = [
    "CLOUD", "FSDS", "T", "U", "V",
    "OMEGA500", "PBLH", "RELHUM", "PRECT"
]

# Emission / chemistry-related variables
cams_cols = [c for c in df.columns if c.startswith("CAMSv51_")]
qfed_cols = [c for c in df.columns if c.startswith("QFED_")]
vsl_cols  = [c for c in df.columns if c.startswith("vsl_")]

# Combine all feature columns
feature_cols = meteo_cols + cams_cols + qfed_cols + vsl_cols

print("Number of features:", len(feature_cols))
print("Feature columns:")
print(feature_cols)



In [ ]:
# ==============================
# 4. Basic data cleaning
# ==============================

df = df.copy()

# Convert datetime column
df[time_col] = pd.to_datetime(df[time_col])

# Drop rows with missing values in target, raw model output, or features
df = df.dropna(subset=[obs_col, raw_col] + feature_cols)

# Define the residual target
# residual = observation - raw model prediction
df["residual"] = df[obs_col] - df[raw_col]

print("Rows after cleaning:", len(df))

In [ ]:
# ==============================
# 5. Prepare X and y
# ==============================

X = df[feature_cols]
y = df["residual"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X,
    y,
    df,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# ==============================
# 6. Train the AutoML model
# ==============================

automl = AutoML()

automl_settings = {
    "time_budget": 60,          # Maximum training time in seconds
    "metric": "rmse",           # Optimisation metric
    "task": "regression",       # Regression task
    "estimator_list": ["lgbm"],  # Use LightGBM only for simplicity
    "seed": 42,
    "verbose": 1,
}

automl.fit(
    X_train=X_train,
    y_train=y_train,
    **automl_settings
)

print("Best model:", automl.best_estimator)
print("Best config:", automl.best_config)

# ==============================
# 7. Predict residuals on the test set
# ==============================

residual_pred = automl.predict(X_test)

# Corrected PM2.5 = raw MUSICA PM2.5 + predicted residual
pm25_raw = df_test[raw_col].values
pm25_obs = df_test[obs_col].values
pm25_corr = pm25_raw + residual_pred

In [ ]:
# ==============================
# 8. Define evaluation functions
# ==============================

def rmse(y_true, y_pred):
    """Calculate root mean squared error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def nmb(y_true, y_pred):
    """Calculate normalized mean bias in percentage."""
    return np.sum(y_pred - y_true) / np.sum(y_true) * 100


# ==============================
# 9. Evaluate raw and corrected PM2.5
# ==============================

results = {
    "RMSE_raw": rmse(pm25_obs, pm25_raw),
    "RMSE_corrected": rmse(pm25_obs, pm25_corr),

    "MAE_raw": mean_absolute_error(pm25_obs, pm25_raw),
    "MAE_corrected": mean_absolute_error(pm25_obs, pm25_corr),

    "R2_raw": r2_score(pm25_obs, pm25_raw),
    "R2_corrected": r2_score(pm25_obs, pm25_corr),

    "NMB_raw_%": nmb(pm25_obs, pm25_raw),
    "NMB_corrected_%": nmb(pm25_obs, pm25_corr),
}

results_df = pd.DataFrame([results])

print("Evaluation results:")
print(results_df.to_string(index=False))



In [ ]:
# ==============================
# 10. Save corrected results
# ==============================

df_test_out = df_test.copy()
df_test_out["residual_pred"] = residual_pred
df_test_out["PM25_corrected"] = pm25_corr

df_test_out.to_csv(
    "corrected_test.csv",
    index=False
)

print("Saved corrected test results.")